# District Heating MILP Playground

This notebook lets you edit a small district-heating dispatch scenario and solve the MILP interactively.

It covers:
- heat pump, boiler, CHP, and thermal storage
- minimum up/down times for boiler and CHP
- CHP startup cost and electricity market revenues
- heat demand loaded from the measured demand CSV in `notebooks/optimization/Data`
- editable day-ahead price time series

The implementation uses `PuLP`. If it is not installed in your environment, the setup cell below will show a short hint.

In [ ]:
from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Any

try:
    import pulp
except ImportError as exc:
    raise ImportError(
        "PuLP is required for this notebook. Install it with `pip install pulp` in your environment."
    ) from exc

try:
    import pandas as pd
except ImportError as exc:
    raise ImportError(
        "pandas is required for this notebook because the heat demand is loaded from CSV. Install it with `pip install pandas`."
    ) from exc

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

print(f"PuLP version: {pulp.__version__}")
print("pandas available:" , pd is not None)
print("matplotlib available:", plt is not None)

## Scenario Inputs

Edit the parameters in this cell to play with the model. By default, the notebook loads hourly measured heat demand from CSV and expands it to 15-minute intervals.

In [ ]:
from pathlib import Path
from IPython.display import display

@dataclass
class Scenario:
    demand_mw: list[float]
    da_price_eur_per_mwh: list[float]
    delta_h: float = 0.25
    storage_capacity_mwh: float = 200.0
    storage_charge_max_mw: float = 15.0
    storage_discharge_max_mw: float = 15.0
    storage_initial_soc_mwh: float = 200.0
    storage_loss_mwh_per_interval: float = 0.000125
    hp_min_power_mwel: float = 1.0
    hp_max_power_mwel: float = 8.0
    hp_cop: float = 3.5
    boiler_min_power_mwth: float = 2.0
    boiler_max_power_mwth: float = 12.0
    boiler_efficiency: float = 0.97
    boiler_min_up_intervals: int = 4
    boiler_min_down_intervals: int = 4
    chp_min_power_mwel: float = 2.0
    chp_max_power_mwel: float = 6.0
    chp_max_heat_mwth: float = 7.0
    chp_efficiency_el: float = 0.40
    chp_efficiency_th: float = 0.48
    chp_efficiency_total: float = 0.88
    chp_min_up_intervals: int = 8
    chp_min_down_intervals: int = 8
    gas_price_eur_per_mwh_hs: float = 35.0
    co2_factor_t_per_mwh_hs: float = 0.201
    co2_price_eur_per_t: float = 60.0
    chp_startup_cost_eur: float = 600.0
    initial_hp_on: int = 0
    initial_boiler_on: int = 0
    initial_chp_on: int = 0

    @property
    def gas_plus_co2_cost_eur_per_mwh_hs(self) -> float:
        return self.gas_price_eur_per_mwh_hs + self.co2_factor_t_per_mwh_hs * self.co2_price_eur_per_t

    @property
    def horizon(self) -> int:
        return len(self.demand_mw)


DATA_PATH_CANDIDATES = [
    Path("Data") / "RawData_MeasuredHeadDemand.csv",
    Path("notebooks") / "optimization" / "Data" / "RawData_MeasuredHeadDemand.csv",
    Path.cwd() / "Data" / "RawData_MeasuredHeadDemand.csv",
    Path.cwd() / "notebooks" / "optimization" / "Data" / "RawData_MeasuredHeadDemand.csv",
]


def resolve_data_path(candidates: list[Path]) -> Path:
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    checked = "\n".join(str(path.resolve()) for path in candidates)
    raise FileNotFoundError(f"Heat demand CSV not found. Checked:\n{checked}")


DATA_PATH = resolve_data_path(DATA_PATH_CANDIDATES)


def load_heat_demand_from_csv(
    csv_path: str | Path = DATA_PATH,
    start: str | None = "2024-03-01 00:00:00+01:00",
    periods: int = 24,
    repeat_each: int = 4,
    target_timezone: str = "Europe/Berlin",
) -> tuple[pd.DataFrame, list[float]]:
    csv_path = Path(csv_path)
    if not csv_path.exists():
        raise FileNotFoundError(f"Heat demand CSV not found: {csv_path.resolve()}")

    df = pd.read_csv(csv_path)
    df["Time Point"] = pd.to_datetime(df["Time Point"], utc=True).dt.tz_convert(target_timezone)
    df = df.sort_values("Time Point").reset_index(drop=True)
    df["heat_demand_mw"] = df["Measured Heat Demand[W]"] / 1_000_000.0

    if start is not None:
        start_ts = pd.Timestamp(start)
        if start_ts.tzinfo is None:
            start_ts = start_ts.tz_localize(target_timezone)
        else:
            start_ts = start_ts.tz_convert(target_timezone)
        df = df[df["Time Point"] >= start_ts].reset_index(drop=True)

    if periods is not None:
        df = df.iloc[:periods].copy()

    if df.empty:
        raise ValueError("No demand data left after applying the selected start/period filters.")

    demand_15min = [value for value in df["heat_demand_mw"].tolist() for _ in range(repeat_each)]
    return df, demand_15min


hourly_demand_df, demand_15min = load_heat_demand_from_csv()
hourly_prices = [65, 60, 55, 50, 48, 45, 50, 65, 85, 95, 100, 92, 80, 72, 68, 70, 78, 88, 105, 110, 98, 85, 75, 68]

prices_15min = [value for value in hourly_prices for _ in range(4)]

scenario = Scenario(
    demand_mw=demand_15min,
    da_price_eur_per_mwh=prices_15min,
)

print("Intervals:", scenario.horizon)
print("Demand source:", DATA_PATH)
print("Loaded hourly demand rows:", len(hourly_demand_df))
print("Selected hourly demand window:", hourly_demand_df["Time Point"].min(), "to", hourly_demand_df["Time Point"].max())
print("Effective gas + CO2 cost [EUR/MWh_Hs]:", round(scenario.gas_plus_co2_cost_eur_per_mwh_hs, 2))
display(hourly_demand_df[["Time Point", "Measured Heat Demand[W]", "heat_demand_mw"]].head(8))

## Model Builder

The CHP is modeled with fuel input plus electric, thermal, and total efficiency constraints. Because of the given CHP data, the thermal cap may implicitly tighten the electrical maximum below 6 MW.

In [ ]:
def build_dispatch_model(s: Scenario) -> tuple[pulp.LpProblem, dict[str, dict[int, pulp.LpVariable]]]:
    if len(s.demand_mw) != len(s.da_price_eur_per_mwh):
        raise ValueError("Demand and price arrays must have the same length.")

    model = pulp.LpProblem("district_heating_dispatch", pulp.LpMinimize)
    T = list(range(s.horizon))
    fuel_cost = s.gas_plus_co2_cost_eur_per_mwh_hs

    p_hp = pulp.LpVariable.dicts("p_hp", T, lowBound=0)
    q_hp = pulp.LpVariable.dicts("q_hp", T, lowBound=0)
    u_hp = pulp.LpVariable.dicts("u_hp", T, lowBound=0, upBound=1, cat="Binary")

    q_boiler = pulp.LpVariable.dicts("q_boiler", T, lowBound=0)
    g_boiler = pulp.LpVariable.dicts("g_boiler", T, lowBound=0)
    u_boiler = pulp.LpVariable.dicts("u_boiler", T, lowBound=0, upBound=1, cat="Binary")
    v_boiler = pulp.LpVariable.dicts("v_boiler", T, lowBound=0, upBound=1, cat="Binary")
    w_boiler = pulp.LpVariable.dicts("w_boiler", T, lowBound=0, upBound=1, cat="Binary")

    p_chp = pulp.LpVariable.dicts("p_chp", T, lowBound=0)
    q_chp = pulp.LpVariable.dicts("q_chp", T, lowBound=0)
    g_chp = pulp.LpVariable.dicts("g_chp", T, lowBound=0)
    u_chp = pulp.LpVariable.dicts("u_chp", T, lowBound=0, upBound=1, cat="Binary")
    v_chp = pulp.LpVariable.dicts("v_chp", T, lowBound=0, upBound=1, cat="Binary")
    w_chp = pulp.LpVariable.dicts("w_chp", T, lowBound=0, upBound=1, cat="Binary")

    charge = pulp.LpVariable.dicts("charge", T, lowBound=0)
    discharge = pulp.LpVariable.dicts("discharge", T, lowBound=0)
    soc = pulp.LpVariable.dicts("soc", T, lowBound=0, upBound=s.storage_capacity_mwh)

    model += (
        pulp.lpSum(
            s.delta_h
            * (
                s.da_price_eur_per_mwh[t] * p_hp[t]
                + fuel_cost * g_boiler[t]
                + fuel_cost * g_chp[t]
                - s.da_price_eur_per_mwh[t] * p_chp[t]
            )
            + s.chp_startup_cost_eur * v_chp[t]
            for t in T
        )
    )

    for t in T:
        model += q_hp[t] + q_boiler[t] + q_chp[t] + discharge[t] - charge[t] == s.demand_mw[t], f"heat_balance_{t}"

        prev_soc = s.storage_initial_soc_mwh if t == 0 else soc[t - 1]
        model += soc[t] == prev_soc + s.delta_h * charge[t] - s.delta_h * discharge[t] - s.storage_loss_mwh_per_interval, f"soc_balance_{t}"
        model += charge[t] <= s.storage_charge_max_mw, f"charge_limit_{t}"
        model += discharge[t] <= s.storage_discharge_max_mw, f"discharge_limit_{t}"

        model += q_hp[t] == s.hp_cop * p_hp[t], f"hp_cop_{t}"
        model += p_hp[t] >= s.hp_min_power_mwel * u_hp[t], f"hp_min_{t}"
        model += p_hp[t] <= s.hp_max_power_mwel * u_hp[t], f"hp_max_{t}"

        model += q_boiler[t] >= s.boiler_min_power_mwth * u_boiler[t], f"boiler_min_{t}"
        model += q_boiler[t] <= s.boiler_max_power_mwth * u_boiler[t], f"boiler_max_{t}"
        model += q_boiler[t] <= s.boiler_efficiency * g_boiler[t], f"boiler_eff_{t}"

        prev_boiler = s.initial_boiler_on if t == 0 else u_boiler[t - 1]
        model += u_boiler[t] - prev_boiler == v_boiler[t] - w_boiler[t], f"boiler_logic_{t}"

        model += p_chp[t] >= s.chp_min_power_mwel * u_chp[t], f"chp_min_el_{t}"
        model += p_chp[t] <= s.chp_max_power_mwel * u_chp[t], f"chp_max_el_{t}"
        model += q_chp[t] <= s.chp_max_heat_mwth * u_chp[t], f"chp_max_th_{t}"
        model += p_chp[t] <= s.chp_efficiency_el * g_chp[t], f"chp_el_eff_{t}"
        model += q_chp[t] <= s.chp_efficiency_th * g_chp[t], f"chp_th_eff_{t}"
        model += p_chp[t] + q_chp[t] <= s.chp_efficiency_total * g_chp[t], f"chp_total_eff_{t}"

        prev_chp = s.initial_chp_on if t == 0 else u_chp[t - 1]
        model += u_chp[t] - prev_chp == v_chp[t] - w_chp[t], f"chp_logic_{t}"

    for t in T:
        if t >= s.boiler_min_up_intervals - 1:
            model += (
                pulp.lpSum(v_boiler[tau] for tau in range(t - s.boiler_min_up_intervals + 1, t + 1)) <= u_boiler[t],
                f"boiler_min_up_{t}",
            )
        if t >= s.boiler_min_down_intervals - 1:
            model += (
                pulp.lpSum(w_boiler[tau] for tau in range(t - s.boiler_min_down_intervals + 1, t + 1)) <= 1 - u_boiler[t],
                f"boiler_min_down_{t}",
            )

        if t >= s.chp_min_up_intervals - 1:
            model += (
                pulp.lpSum(v_chp[tau] for tau in range(t - s.chp_min_up_intervals + 1, t + 1)) <= u_chp[t],
                f"chp_min_up_{t}",
            )
        if t >= s.chp_min_down_intervals - 1:
            model += (
                pulp.lpSum(w_chp[tau] for tau in range(t - s.chp_min_down_intervals + 1, t + 1)) <= 1 - u_chp[t],
                f"chp_min_down_{t}",
            )

    variables = {
        "p_hp": p_hp,
        "q_hp": q_hp,
        "u_hp": u_hp,
        "q_boiler": q_boiler,
        "g_boiler": g_boiler,
        "u_boiler": u_boiler,
        "v_boiler": v_boiler,
        "w_boiler": w_boiler,
        "p_chp": p_chp,
        "q_chp": q_chp,
        "g_chp": g_chp,
        "u_chp": u_chp,
        "v_chp": v_chp,
        "w_chp": w_chp,
        "charge": charge,
        "discharge": discharge,
        "soc": soc,
    }
    return model, variables

## Solve and Inspect Results

In [ ]:
def solve_dispatch(s: Scenario, msg: bool = True) -> tuple[pulp.LpProblem, dict[str, dict[int, pulp.LpVariable]], dict[str, Any]]:
    model, variables = build_dispatch_model(s)
    solver = pulp.PULP_CBC_CMD(msg=msg)
    status_code = model.solve(solver)
    status = pulp.LpStatus[status_code]

    summary = {
        "status": status,
        "objective_eur": pulp.value(model.objective),
        "intervals": s.horizon,
        "fuel_cost_eur_per_mwh_hs": s.gas_plus_co2_cost_eur_per_mwh_hs,
    }
    return model, variables, summary


def extract_results(s: Scenario, variables: dict[str, dict[int, pulp.LpVariable]]):
    rows = []
    for t in range(s.horizon):
        row = {
            "t": t,
            "hour": t * s.delta_h,
            "price_eur_per_mwh": s.da_price_eur_per_mwh[t],
            "demand_mw": s.demand_mw[t],
            "p_hp_mwel": pulp.value(variables["p_hp"][t]),
            "q_hp_mwth": pulp.value(variables["q_hp"][t]),
            "boiler_on": pulp.value(variables["u_boiler"][t]),
            "q_boiler_mwth": pulp.value(variables["q_boiler"][t]),
            "chp_on": pulp.value(variables["u_chp"][t]),
            "p_chp_mwel": pulp.value(variables["p_chp"][t]),
            "q_chp_mwth": pulp.value(variables["q_chp"][t]),
            "charge_mwth": pulp.value(variables["charge"][t]),
            "discharge_mwth": pulp.value(variables["discharge"][t]),
            "soc_mwh": pulp.value(variables["soc"][t]),
        }
        rows.append(row)

    if pd is not None:
        return pd.DataFrame(rows)
    return rows


model, variables, summary = solve_dispatch(scenario, msg=False)
summary

In [ ]:
results = extract_results(scenario, variables)

if pd is not None:
    display(results.head(16))
    display(results[["demand_mw", "q_hp_mwth", "q_boiler_mwth", "q_chp_mwth", "discharge_mwth", "charge_mwth", "soc_mwh"]].describe())
else:
    results[:5]

In [ ]:
if pd is not None:
    total_hp_power_mwh = (results["p_hp_mwel"] * scenario.delta_h).sum()
    total_chp_power_mwh = (results["p_chp_mwel"] * scenario.delta_h).sum()
    total_boiler_heat_mwh = (results["q_boiler_mwth"] * scenario.delta_h).sum()
    total_chp_starts = results["chp_on"].diff().fillna(results["chp_on"]).clip(lower=0).sum()
    print(f"Objective [EUR]: {summary['objective_eur']:.2f}")
    print(f"Total HP electricity [MWh]: {total_hp_power_mwh:.2f}")
    print(f"Total CHP electricity sold [MWh]: {total_chp_power_mwh:.2f}")
    print(f"Total boiler heat [MWh_th]: {total_boiler_heat_mwh:.2f}")
    print(f"CHP starts: {total_chp_starts:.0f}")
else:
    print(summary)

In [ ]:
if pd is not None and plt is not None:
    fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)

    axes[0].plot(results["hour"], results["demand_mw"], label="Total heat demand to be served [MW_th]", linewidth=2.5, color="black")
    axes[0].plot(results["hour"], results["q_hp_mwth"], label="Heat produced by the heat pump [MW_th]", linewidth=2)
    axes[0].plot(results["hour"], results["q_boiler_mwth"], label="Heat produced by the gas boiler [MW_th]", linewidth=2)
    axes[0].plot(results["hour"], results["q_chp_mwth"], label="Heat produced by the combined heat and power unit [MW_th]", linewidth=2)
    axes[0].plot(results["hour"], results["discharge_mwth"] - results["charge_mwth"], label="Net thermal storage output: discharge minus charge [MW_th]", linewidth=2)
    axes[0].set_title("Heat Balance Over Time", fontsize=13)
    axes[0].set_ylabel("Thermal power [MW_th]")
    axes[0].legend(loc="upper right")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(results["hour"], results["price_eur_per_mwh"], color="tab:orange", label="Day-ahead electricity market price [EUR/MWh_el]", linewidth=2.5)
    axes[1].plot(results["hour"], results["p_chp_mwel"], color="tab:green", label="Electricity sold by the CHP unit [MW_el]", linewidth=2)
    axes[1].plot(results["hour"], results["p_hp_mwel"], color="tab:blue", label="Electricity consumed by the heat pump [MW_el]", linewidth=2)
    axes[1].set_title("Electricity Market Interaction", fontsize=13)
    axes[1].set_ylabel("Electrical power / market price")
    axes[1].legend(loc="upper right")
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(results["hour"], results["soc_mwh"], color="tab:red", label="Thermal energy stored in the buffer tank [MWh_th]", linewidth=2.5)
    axes[2].set_title("Thermal Storage State of Charge", fontsize=13)
    axes[2].set_ylabel("Stored thermal energy [MWh_th]")
    axes[2].set_xlabel("Time in model horizon [hours]")
    axes[2].legend(loc="upper right")
    axes[2].grid(True, alpha=0.3)

    fig.suptitle("District Heating MILP Dispatch: Demand Coverage, Electricity Trading, and Storage Utilization", fontsize=15)
    plt.tight_layout()
    plt.show()
else:
    print("Install pandas and matplotlib to get tabular output and plots.")

## Ideas to Try

- change the day-ahead price profile and see when CHP operation becomes profitable
- lower the storage initial state of charge and observe the dispatch change
- increase the heat demand peak and check which asset becomes binding
- compare cases with higher CO2 prices or lower heat pump COP

If you want, the next step can be to refactor this notebook into reusable Python modules under `src/` so the model and the notebook share the same code.